# v3 — the patch

v1 and v2 were faithful reproductions with their defects corrected. This is the
step where the design is allowed to *change*. Four pieces:

| | |
|---|---|
| **A1** | the player metric — the per-pitch score becomes a swappable component and the choice is made on evidence |
| **A2** | pitch-frame features: batter-frame location, handedness, velocity and movement |
| **A3** | the take model — verify what it is actually learning |
| **A4** | personalization: replace the binary hot-zone flag with a continuous surface |

Each step changes one thing, so every delta is attributable. Every figure is
held out: each of 2023, 2024 and 2025 is scored by a fold model that never saw
it. Generic variants use the generic folds (training from 2021); whenever a
personalized variant is involved, every row uses the personalized folds
(training from 2022, 2021 as prior data only), so the comparison does not vary
the training data along with the feature. 2026 is not loaded: it is kept out of
model selection (earlier versions scored it, so it is no longer a clean test).

In [1]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import xgboost as xgb

from src import data as D, evaluate as E, features as F, decision as DEC
from src.baselines import predict_both, V1_FEATURES, _dmatrix

pd.set_option('display.width', 160); sns.set_theme(style='whitegrid')
PALETTE = ['#FFB000','#648FFF','#785EF0','#DC267F','#FE6100','#3D1EB2']
SEASONS = range(2021, 2026)          # 2026 is kept out of model selection; not loaded here
SHOW = E.SHOW
SUB = ['swing signal recovered', 'swing calib slope', 'take signal recovered', 'take calib slope']

# the full pitch frame, used from A2 onward
V3_FEATURES = ['plate_x_bat','plate_z_norm','count','stand','p_throws',
               'release_speed','pfx_x','pfx_z','pitch_type']

In [2]:
df = D.drop_pitchers_batting(D.load_seasons(SEASONS, verbose=False))
df = D.add_zone_frame(df.dropna(subset=['plate_x_mid','plate_z_mid']))
for c in ('stand','p_throws','pitch_type'):
    df[c] = df[c].astype('category')
print(f'{len(df):,} pitches')

3,517,807 pitches


## A1 — which per-pitch score?

Every candidate is built from the same counterfactual pair, `q_swing` and
`q_take`, so the models are identical and only the aggregation differs. The
models here are v1's features on the generic folds, held fixed.

- `chosen_value` — the value of the action taken
- `signed_edge` — how much better the chosen action was than the alternative
- `regret` — `max(0, −signed_edge)`; zero whenever the hitter was right
- `close_weighted` — signed correctness, weighted by how close the call was
- `correct_decision` — +1 right, −1 wrong, no magnitude at all

**The selection criterion, fixed before the numbers:** construct validity first
(does it punish chasing *and* reward attacking strikes, each partialled on the
other, since the two correlate +0.5 through aggression); reliability second as
a floor; Zone% as a veto above |r| ≈ 0.3; predictive validity reported but not
decisive, because it asks whether the metric predicts future *production*,
which is mostly hitting ability rather than judgement.

In [3]:
run_a1 = E.run_folds(df, V1_FEATURES, E.GENERIC_FOLDS)
a1 = pd.DataFrame([{'variant': name, **E.metric_checks(run_a1.held, name)['summary']}
                   for name in DEC.SCORES]).set_index('variant')
a1[SHOW]

,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,
chosen_value,-0.884,0.673,0.758,0.526,0.082,0.135
signed_edge,-0.921,0.701,0.819,0.594,0.301,0.085
regret,-0.825,0.527,0.791,0.572,0.504,0.027
close_weighted,-0.717,0.830,0.641,0.345,0.050,0.098
correct_decision,-0.954,0.914,0.825,0.586,0.241,0.113


In [4]:
# close_weighted's kernel scale controls how narrowly it focuses on close calls.
h = run_a1.held
rows = []
for scale in [0.02, 0.05, 0.10, 0.20, 0.40, 0.80]:
    col = f'cw_{scale}'
    h[col] = DEC.close_weighted(h, scale=scale)
    rows.append({'scale': scale, **E.metric_checks(h, col)['summary']})
sweep = pd.DataFrame(rows).set_index('scale')
sc_cw, sc_cd = E.player_metric(h, 'cw_0.8'), E.player_metric(h, 'correct_decision')
m = sc_cw.merge(sc_cd, on=['season', 'batter'])
print(f'corr(close_weighted at 0.8, correct_decision) across hitter-seasons: '
      f'{np.corrcoef(m.raw_x, m.raw_y)[0, 1]:.3f}')
sweep[SHOW]

corr(close_weighted at 0.8, correct_decision) across hitter-seasons: 0.997


,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
scale,,,,,,
0.02,-0.292,0.636,0.359,0.137,0.107,0.063
0.05,-0.717,0.830,0.641,0.345,0.050,0.098
0.10,-0.861,0.876,0.730,0.446,0.111,0.112
0.20,-0.917,0.899,0.779,0.512,0.176,0.116
0.40,-0.939,0.910,0.804,0.550,0.210,0.116
0.80,-0.948,0.913,0.816,0.568,0.226,0.115


In [5]:
# What each score pays out, by how obvious the decision is (runs per pitch).
def region(g):
    inz = D.in_rulebook_zone(g, ball_edge=True)
    near = (g.plate_x_bat.abs() <= 1.15) & g.plate_z_norm.between(-0.35, 1.35)
    return pd.Series(np.where(inz, 'zone', np.where(near, 'shadow', 'chase')), index=g.index)

h['region'] = region(h)
h.groupby('region', observed=True)[['signed_edge', 'regret']].mean().round(4)

,signed_edge,regret
region,,
chase,0.0818,-0.0256
shadow,0.0104,-0.0321
zone,0.0495,-0.0157


**By the criterion fixed in advance, `correct_decision` wins.** It has the best
construct validity by a wide margin (−0.954 / +0.914, against −0.921 / +0.701
for `signed_edge`), the best split-half reliability, and passes the Zone% veto
(0.241) that `signed_edge` sits right on (0.301) and `regret` fails outright
(0.504). The sweep shows why: as the closeness kernel widens every weight
approaches 1, `close_weighted` converges on `correct_decision` (correlation
0.997 at scale 0.8), and construct validity improves the whole way.

**The run-value magnitude is what carries the pitch-mix bias.** `signed_edge`
pays about eight times more per pitch for an obvious take (chase region, +0.082
runs) than for a genuinely close call (shadow, +0.010), so a hitter thrown more
junk scores higher for the same judgement. `regret` has the mirror-image
problem: its largest losses are hittable pitches taken, so more strikes mean
more regret. Dropping the magnitude removes both.

**But dropping it has a cost.** A sign-based score treats a razor-thin call and
an obvious blunder alike, and it cannot register a feature that changes how much
a swing is worth without flipping which action is better. `signed_edge` stays
the selected score for that reason — every published metric keeps the
magnitude — with its contamination a documented, unsolved limitation. A4 and
the ladder below measure what each choice gives up.

## A2 — pitch-frame features

Added in three groups so each is attributable, on the generic folds, scored
with `signed_edge`. Location moves to the batter frame (positive = inside for
every hitter) and height is normalised against the common zone from listed
height rather than the per-pitch operator bounds, which carry 0.073–0.098 ft of
measurement noise and change definition in 2026.

In [6]:
LADDER = {
  'v1 features':            V1_FEATURES,
  '+batter frame':          ['plate_x_bat','plate_z_norm','count'],
  '+handedness':            ['plate_x_bat','plate_z_norm','count','stand','p_throws'],
  '+pitch characteristics': V3_FEATURES,
}
a2_runs = {label: E.run_folds(df, feats, E.GENERIC_FOLDS) for label, feats in LADDER.items()}
a2_res = {label: E.harness(run, label) for label, run in a2_runs.items()}
a2 = pd.DataFrame([r['summary'] for r in a2_res.values()]).set_index('variant')
a2[SUB + SHOW]

,swing signal recovered,swing calib slope,take signal recovered,take calib slope,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,,,,,
v1 features,0.838,1.261,0.992,1.027,-0.921,0.701,0.819,0.594,0.301,0.085
+batter frame,0.918,1.234,0.991,1.017,-0.925,0.720,0.820,0.596,0.278,0.103
+handedness,0.916,1.236,0.992,1.017,-0.924,0.719,0.820,0.597,0.276,0.100
+pitch characteristics,0.897,1.254,0.991,1.020,-0.923,0.723,0.818,0.593,0.272,0.103


In [7]:
# The sub-models on the conditional means they estimate, per held-out season.
pd.concat({label: r['calibration']['swing'] for label, r in a2_res.items()},
          names=['features']).round(4)

bins  pitches     err  err_count_only   noise  signal_recovered   slope
features               season                                                                         
v1 features            2023     603   232478  0.0214          0.0378  0.0162            0.8314  1.3260
                       2024     595   231511  0.0205          0.0354  0.0158            0.8322  1.2493
                       2025     596   232635  0.0199          0.0345  0.0160            0.8490  1.2064
+batter frame          2023     603   232478  0.0195          0.0378  0.0162            0.8982  1.2940
                       2024     595   231511  0.0185          0.0354  0.0158            0.9098  1.2146
                       2025     596   232635  0.0175          0.0345  0.0160            0.9448  1.1924
+handedness            2023     603   232478  0.0196          0.0378  0.0162            0.8946  1.2978
                       2024     595   231511  0.0185          0.0354  0.0158            0.9096  1.2160
                       2025     596   232635  0.0176          0.0345  0.0160            0.9429  1.1949
+pitch characteristics 2023     603   232478  0.0202          0.0378  0.0162            0.8731  1.3184
                       2024     595   231511  0.0189          0.0354  0.0158            0.8937  1.2355
                       2025     596   232589  0.0181          0.0344  0.0160            0.9238  1.2068

In [8]:
# For contrast, pitch-level RMSE against the same count-only predictor.
pd.concat({label: r['sub_models']['improvement_%'] for label, r in a2_res.items()},
          axis=1).round(2)

,v1 features,+batter frame,+handedness,+pitch characteristics
action,,,,
take,42.78,44.99,45.19,45.39
swing,0.80,0.85,0.85,0.88


**Pitch-level RMSE says the features do almost nothing for the swing model**:
0.80% better than count-only with v1's features, 0.88% with the full pitch
frame. The take model gains steadily (42.8% → 45.4%): what follows a take is an
umpire's call, and location predicts it well.

**On the conditional means, the batter frame is a real gain.** Mirroring
location and normalising height moves the swing model from 83–85% to 90–94% of
the between-bin structure the count-only predictor misses, the largest
sub-model improvement in the ladder. Handedness adds nothing on top.

**Pitch characteristics cannot be judged by this check.** The bins are
location × count, so a feature that explains variation *within* a bin — a
slider and a fastball at the same spot — earns no credit here, and pitch
characteristics slightly lower the share (0.916 → 0.897). The whiff model below
is where they show up.

**The slope is above 1 at every step** (1.19–1.33): across location × count
bins, observed means spread more than the predictions, so the model appears to
understate the location contrast in swing value. No feature changes it, which
points at the hyperparameters held fixed across the ladder (learning rate 0.01
over 200 rounds). The check sees only location and count, and the slope has no
confidence interval yet.

At the player-metric level the pitch frame buys a little: next-season validity
0.085 → 0.103 and Zone% 0.301 → 0.272, with construct validity flat.

### And the events are predictable even though the run value is not

This is the case for decomposing a swing into whiff / foul / in-play rather
than regressing run value directly. Whether a batted ball falls in is close to
luck; whether the hitter *misses* is not. Same generic folds.

In [9]:
pd.concat({'location + count': E.whiff_auc(df, V1_FEATURES),
           'full pitch frame': E.whiff_auc(df, V3_FEATURES)}, names=['features']).round(4)

swings     auc  logloss  logloss_base_rate  improvement_%
features         season                                                           
location + count 2023    339208  0.7305   0.4882             0.5709        14.4868
                 2024    337782  0.7322   0.4837             0.5659        14.5261
                 2025    336956  0.7345   0.4810             0.5650        14.8752
full pitch frame 2023    339208  0.7655   0.4699             0.5709        17.6886
                 2024    337782  0.7682   0.4646             0.5659        17.8958
                 2025    336884  0.7701   0.4618             0.5650        18.2712

**Whiff probability predicts at AUC 0.77 held out**, and velocity and movement
add clearly over location and count in every season (AUC 0.73 → 0.77; log loss
14.5–14.9% → 17.7–18.3% better than the base rate). The intermediate events
carry signal that a direct run-value regression has to infer through a far
noisier target. That is the argument for the event decomposition, and it
survives.

## A3 — what is the take model actually learning?

A take can only end three ways, and the target is the league run value of
`(outcome, count)`. Given the count, the only thing location can tell the model
is the probability of a called strike. If that is right, then `take_pred` should
be an exact affine function of `P(called strike)` within each count, with slope
`RE(called_strike, c) − RE(ball, c)`.

Checked on fold 3: models trained on 2021–24, scored on held-out 2025.

In [10]:
TR3, VA3 = E.GENERIC_FOLDS[-1]
rv3 = D.run_value_table(df, TR3)
is_take = lambda d: ~d.swing & d.outcome.isin(['ball','called_strike'])
tr = df[df.season.isin(TR3) & is_take(df)]
cs_model = xgb.train(
    {'objective':'binary:logistic','eval_metric':'logloss','max_depth':8,
     'learning_rate':0.05,'tree_method':'hist','random_state':1126},
    _dmatrix(tr, V1_FEATURES, (tr.outcome == 'called_strike').astype(int)), 300)

va = run_a1.held[(run_a1.held.season == VA3) & is_take(run_a1.held)].copy()
va['p_cs'] = cs_model.predict(_dmatrix(va, V1_FEATURES))

rows = []
for c, g in va.groupby('count', observed=True):
    if len(g) < 2000: continue
    r = np.corrcoef(g.q_take, g.p_cs)[0, 1]
    rows.append({'count': c, 'n': len(g), 'R2': r**2,
                 'fitted slope': np.polyfit(g.p_cs, g.q_take, 1)[0],
                 'RE(CS)-RE(ball)': rv3.get(('called_strike', c)) - rv3.get(('ball', c))})
a3 = pd.DataFrame(rows).set_index('count')
print(f'median R2 = {a3.R2.median():.4f}   (threshold 0.95)')
a3.round(4)

median R2 = 0.9914   (threshold 0.95)


,n,R2,fitted slope,RE(CS)-RE(ball)
count,,,,
0-0,122883,0.9761,-0.0802,-0.0781
0-1,46317,0.9404,-0.0882,-0.0849
0-2,23586,0.9523,-0.1948,-0.1892
1-0,39077,0.9923,-0.1114,-0.1106
1-1,31957,0.9770,-0.1156,-0.1136
1-2,29434,0.9719,-0.2339,-0.2299
2-0,13594,0.9972,-0.1725,-0.1731
2-1,15068,0.9946,-0.1799,-0.1792
2-2,20944,0.9909,-0.3267,-0.3259


**Confirmed on a held-out season.** Median R² is 0.991 (lowest 0.940, on 0-1)
and the fitted slopes match the run-value gaps to within a few thousandths — on
3-2, −0.611 against an expected −0.614.

The take model is a called-strike probability wearing a regression's clothes. It
learns `P(CS | location)` from the data and multiplies it by run values it reads
off the `count` feature. Two consequences:

1. There is nothing to fix here. Replacing it with the explicit structural form
   `Q_take = P(CS)·RE(CS,c) + (1−P(CS))·RE(ball,c)` would produce the same
   numbers, so the substitution is optional rather than a correction.
2. A hitter-specific contact feature cannot help this model, because nothing
   about how hard a hitter hits changes an umpire's call. A4 tests that.

## A4 — personalization

The binary hot-zone flag is replaced by a continuous surface: the hitter's
expected exit velocity at the pitch location, kernel-smoothed and shrunk toward
the league by effective sample size. Measured year over year the surface
reproduces itself at r ≈ 0.66 against ≈ 0.30 for raw bins (EDA §6), so it
should be a far better estimator of the same signal.

Three rules, each from an earlier result:

- **Fixed two-season prior window** for both the hull and the surface. The
  expanding window made the hull grow with accumulated history (v2), and two
  seasons is where the surface reproduces itself best (EDA §6).
- **Swing model only.** A3 shows the take model is a called-strike
  probability, and v2 measured the hull doing nothing for it. Contact quality
  cannot change an umpire's call. Letting the feature into the take model
  anyway would let `Q_take` vary by hitter, so part of any gain could arrive
  through the wrong branch — reported below both ways, to see.
- **Personalized folds for every row**, generic rows included.

In [11]:
nitro, coverage = F.season_in_nitro(df, window=F.PRIOR_WINDOW)
hot = F.season_hot_zone(df)
key = ['game_pk', 'at_bat_number', 'pitch_number']
pdf = hot.merge(nitro[key + ['in_nitro']], on=key, how='left', validate='one_to_one')
assert pdf.in_nitro.notna().all()

print('feature stability across seasons (the drift check):')
pd.concat([pdf.groupby('season').in_nitro.mean().rename('in_nitro rate'),
           pdf.groupby('season').hot_zone.mean().rename('hot_zone mean'),
           coverage.coverage.rename('hull coverage')], axis=1).round(3)

feature stability across seasons (the drift check):


,in_nitro rate,hot_zone mean,hull coverage
season,,,
2022,0.145,88.772,0.793
2023,0.203,88.640,0.824
2024,0.207,88.684,0.866
2025,0.211,88.811,0.842


In [12]:
F0 = ['plate_x_bat','plate_z_norm','count']
A4 = {
  'no personalization':               (F0, True),
  '+binary hull (swing only)':        (F0 + ['in_nitro'], True),
  '+continuous surface (swing only)': (F0 + ['hot_zone'], True),
  '+continuous surface (both models)':(F0 + ['hot_zone'], False),
}
a4_runs = {label: E.run_folds(pdf, feats, E.PERSONALIZED_FOLDS, swing_only=so)
           for label, (feats, so) in A4.items()}
a4 = pd.DataFrame([E.harness(run, label)['summary'] for label, run in a4_runs.items()]).set_index('variant')
a4[SUB + SHOW]

,swing signal recovered,swing calib slope,take signal recovered,take calib slope,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,,,,,
no personalization,0.906,1.244,0.994,1.012,-0.925,0.720,0.820,0.594,0.279,0.103
+binary hull (swing only),0.914,1.231,0.994,1.012,-0.919,0.709,0.820,0.592,0.266,0.106
+continuous surface (swing only),0.910,1.258,0.994,1.012,-0.895,0.670,0.841,0.629,0.185,0.157
+continuous surface (both models),0.910,1.258,0.994,1.012,-0.894,0.669,0.843,0.629,0.181,0.157


**The fixed window removes the drift.** The hull flag fires on 20.3–21.1% of
pitches from 2023 on (2022 has only one prior season), where the expanding
window in v2 climbed to 25.1%. The surface's mean is stable at 88.6–88.8 mph.

- **The binary hull buys nothing**, even de-leaked, on a fixed window and in the
  swing model only: every column moves by 0.013 or less.
- **The continuous surface is the largest feature effect found.** Split-half
  0.820 → 0.841, YoY R² 0.594 → 0.629, Zone% contamination down a third
  (0.279 → 0.185), next-season validity up by half (0.103 → 0.157). It costs
  some construct validity: −0.925 / +0.720 → −0.895 / +0.670.
- **Swing-only and both-models are the same**, within 0.004 on every column,
  and the take model's calibration does not move. None of the gain came through
  the take branch.
- **The sub-model check cannot see it** (swing share 0.906 → 0.910), for the
  same reason it cannot see pitch characteristics: the bins pool hitters, and
  the surface varies *between* hitters within a bin.

### Does the hot zone actually differ between hitters?

"Redundant with location" would mean every hitter's hot zone is the same
region — in which case the model already knows it from `plate_x_bat` and
`plate_z_norm`. Check the premise directly.

In [13]:
inz = D.in_rulebook_zone(pdf, ball_edge=True)
z = pdf[inz].assign(
    cx=pd.cut(pdf[inz].plate_x_bat, np.linspace(-0.85, 0.85, 5), labels=False),
    cz=pd.cut(pdf[inz].plate_z_norm, np.linspace(0, 1, 5), labels=False)).dropna(subset=['cx','cz'])
cellwise = z.groupby(['season','cx','cz','batter'], observed=True).hot_zone.mean().rename('v').reset_index()
spread = cellwise.groupby(['season','cx','cz'], observed=True).v.agg(['mean','std','size']).query('size >= 100')

print(f'between-hitter SD within one zone cell : {spread["std"].median():.2f} mph')
print(f'league mean across cells               : {spread["mean"].min():.1f} - {spread["mean"].max():.1f} mph')
print(f'  -> personalization is {spread["std"].median()/(spread["mean"].max()-spread["mean"].min()):.1f}x the size of the location effect\n')

best = cellwise.groupby(['season','batter'], observed=True).apply(
    lambda g: g.loc[g.v.idxmax(), ['cx','cz']], include_groups=False)
print("where each hitter's best cell falls (share of hitter-seasons):")
(best.groupby(['cx','cz']).size() / len(best)).unstack().round(3)

between-hitter SD within one zone cell : 1.72 mph
league mean across cells               : 87.3 - 90.2 mph
  -> personalization is 0.6x the size of the location effect



where each hitter's best cell falls (share of hitter-seasons):


cz,0.0,1.0,2.0,3.0
cx,,,,
0.0,0.194,0.027,0.036,0.035
1.0,0.045,0.146,0.211,0.075
2.0,0.063,0.097,0.042,0.008
3.0,0.015,0.003,0.002,0.002


**The premise fails.** Hitters differ at the same location by 1.72 mph — about
0.6× the entire league-wide location effect (87.3–90.2 mph) — and they differ
in *where* their best region is: the most common best cell holds only 21% of
hitter-seasons, with real mass spread across five or six. Hitters cannot cover
the whole zone, and they do not cover the same part of it.

In [14]:
# How much does the feature move the counterfactual, and how often does it flip
# the recommended action? Fold 3 models, held-out 2025.
va = pdf[pdf.season == 2025]
d_no  = predict_both(va, a4_runs['no personalization'].models[2025])
d_hot = predict_both(va, a4_runs['+continuous surface (swing only)'].models[2025])

dq = d_hot.q_swing - d_no.q_swing
flip = np.sign(d_no.edge) != np.sign(d_hot.edge)
print(f'change in Q_swing from the feature : SD {dq.std():.4f} runs  (Q_swing itself: SD {d_no.q_swing.std():.4f})')
print(f'change in Q_take                   : max |.| {(d_hot.q_take - d_no.q_take).abs().max():.4f} runs  (take model has no hot_zone)')
print(f'pitches where the RECOMMENDED ACTION flips: {flip.mean():.2%}')

change in Q_swing from the feature : SD 0.0104 runs  (Q_swing itself: SD 0.0406)
change in Q_take                   : max |.| 0.0000 runs  (take model has no hot_zone)
pitches where the RECOMMENDED ACTION flips: 1.58%


**A large effect that rarely flips the decision.** The surface moves `Q_swing`
with a standard deviation of 0.010 runs, a quarter of `Q_swing`'s own (0.041),
but flips the *recommended action* on only 1.6% of pitches, and `Q_take` does
not move at all. A sign-based score registers the feature on those 1.6% and
nowhere else.

In [15]:
# The feature against every metric, models held fixed.
rows = []
for feat_label, run_label in [('without hot_zone', 'no personalization'),
                              ('with hot_zone',    '+continuous surface (swing only)')]:
    for score in DEC.SCORES:
        rows.append({'metric': score, 'features': feat_label,
                     **E.metric_checks(a4_runs[run_label].held, score)['summary']})
inter = pd.DataFrame(rows).set_index(['metric','features'])
inter

,,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
metric,features,,,,,,
chosen_value,without hot_zone,-0.884,0.671,0.752,0.523,0.080,0.145
signed_edge,without hot_zone,-0.925,0.720,0.820,0.594,0.279,0.103
regret,without hot_zone,-0.850,0.578,0.776,0.549,0.494,0.051
close_weighted,without hot_zone,-0.737,0.828,0.625,0.298,0.042,0.114
correct_decision,without hot_zone,-0.965,0.933,0.827,0.580,0.224,0.137
chosen_value,with hot_zone,-0.620,0.418,0.901,0.699,0.186,0.286
signed_edge,with hot_zone,-0.895,0.670,0.841,0.629,0.185,0.157
regret,with hot_zone,-0.787,0.462,0.797,0.556,0.551,-0.012
close_weighted,with hot_zone,-0.656,0.766,0.654,0.296,0.110,0.064


**Personalization interacts with the metric.**

- Under `chosen_value` it is transformative on reliability and prediction
  (split-half 0.752 → 0.901, YoY 0.523 → 0.699, next-season 0.145 → **0.286**)
  but construct validity collapses (−0.884 / +0.671 → −0.620 / +0.418). The
  score drifts from "did he decide well" toward "is he a good hitter", which is
  also why it predicts production so much better.
- Under `signed_edge` reliability rises, Zone% contamination falls by a third
  and next-season validity by half, while construct validity gives back
  0.03–0.05.
- Under `correct_decision` it does little: construct validity and reliability
  within 0.003, next-season 0.137 → 0.150.
- `regret` and `close_weighted` get worse.

**The metric trade, held out.** `correct_decision` still has by far the best
construct validity (−0.964 / +0.931 with the surface) and next-season validity
close to `signed_edge`'s (0.150 against 0.157). `signed_edge` with the surface
is more reliable (split-half 0.841 against 0.828, YoY 0.629 against 0.583) and
less contaminated (Zone% 0.185 against 0.215). The held-out folds show the same
trade the earlier in-sample comparison did; they do not reverse it.

## The ladder at a fixed metric

Everything above varies one thing at a time *within* a step, but the steps
themselves changed two things at once: v1 and v2 score `chosen_value` because
that is their design, while this notebook changed the metric as well as the
features. A v1→v3 difference read off those rows would confound the two.

So run the whole feature ladder with the metric held fixed, on the personalized
folds, and report it under both candidate metrics. Each row is a complete
feature set, not an increment: the hull and the surface estimate the same thing
— a hitter's hot zone — so they are never combined. Hitter features go to the
swing model only, except in the last row.

In [16]:
LADDER = {
  'v1  location+count':                  (V1_FEATURES, True),
  'v2  +nitro hull':                     (V1_FEATURES + ['in_nitro'], True),
  'v3  full pitch frame':                (V3_FEATURES, True),
  'v3  +hot-zone surface':               (V3_FEATURES + ['hot_zone'], True),
  'v3  +hot-zone surface (both models)': (V3_FEATURES + ['hot_zone'], False),
}
ladder_runs = {label: E.run_folds(pdf, feats, E.PERSONALIZED_FOLDS, swing_only=so,
                                  in_sample=(label == 'v3  +hot-zone surface'))
               for label, (feats, so) in LADDER.items()}
rows = []
for score in ['chosen_value', 'signed_edge']:
    for label, run in ladder_runs.items():
        rows.append({'metric': score, 'features': label, **E.harness(run, label, score)['summary']})
ladder = pd.DataFrame(rows).drop(columns='variant').set_index(['metric','features'])
ladder[SHOW]

chase | zone_swing  zone_swing | chase  split-half r  YoY R2 (mean)  Zone% |r|  next-season r (partial)
metric       features                                                                                                                                    
chosen_value v1  location+count                               -0.883               0.669         0.757          0.526      0.081                    0.137
             v2  +nitro hull                                  -0.797               0.544         0.792          0.512      0.028                    0.138
             v3  full pitch frame                             -0.878               0.657         0.751          0.511      0.086                    0.150
             v3  +hot-zone surface                            -0.625               0.419         0.900          0.693      0.180                    0.289
             v3  +hot-zone surface (both models)              -0.625               0.419         0.900          0.694      0.180                    0.288
signed_edge  v1  location+count                               -0.920               0.699         0.818          0.594      0.300                    0.082
             v2  +nitro hull                                  -0.913               0.688         0.819          0.589      0.282                    0.086
             v3  full pitch frame                             -0.923               0.723         0.817          0.590      0.274                    0.107
             v3  +hot-zone surface                            -0.895               0.677         0.838          0.624      0.184                    0.154
             v3  +hot-zone surface (both models)              -0.892               0.671         0.841          0.625      0.176                    0.157

In [17]:
# The sub-models, which do not depend on the metric.
ladder.loc['signed_edge', SUB]

,swing signal recovered,swing calib slope,take signal recovered,take calib slope
features,,,,
v1 location+count,0.827,1.279,0.993,1.019
v2 +nitro hull,0.844,1.276,0.993,1.019
v3 full pitch frame,0.886,1.271,0.994,1.014
v3 +hot-zone surface,0.892,1.291,0.994,1.014
v3 +hot-zone surface (both models),0.892,1.291,0.994,1.012


**Read down the `signed_edge` block.** From v1's features to the full pitch
frame with the surface, next-season validity nearly doubles (0.082 → 0.154),
Zone% contamination falls 39% (0.300 → 0.184), and reliability rises
(split-half 0.818 → 0.838, YoY R² 0.594 → 0.624). Construct validity slips,
−0.920 / +0.699 → −0.895 / +0.677. The hull contributes nothing; the pitch
frame lifts next-season validity (0.082 → 0.107) with construct validity flat;
the surface supplies the rest, and all of the construct cost.

**The `chosen_value` block shows why the metric matters.** There the surface
takes next-season validity from 0.150 to 0.289 while construct validity
collapses (−0.878 / +0.657 → −0.625 / +0.419). `signed_edge` takes a smaller
gain and keeps most of the construct.

**The surface in both models matches swing-only** to within 0.008 on every
column. No personalization gain came through the take side.

## v3

Full pitch frame plus the continuous hot-zone surface, routed to the swing
model only, scored with `signed_edge`, on the personalized folds. Swing-only
rather than both models: the two measure the same, and A3 says the take model
cannot use the feature.

The in-sample row scores fold 3's model on its own training seasons, 2022–24,
so the optimism of in-sample scoring is visible. It is not averaged into
anything.

In [18]:
v3_run = ladder_runs['v3  +hot-zone surface']
v3 = E.harness(v3_run, 'v3')
ins = E.in_sample_checks(v3_run, 'v3')
pd.DataFrame([v3['summary'], ins['summary']]).set_index('variant')[SHOW]

,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial)
variant,,,,,,
v3,-0.895,0.677,0.838,0.624,0.184,0.154
v3 (in-sample),-0.905,0.684,0.839,0.643,0.223,0.165


In [19]:
try:
    from pybaseball import playerid_reverse_lookup
    s25 = v3['scores'].query('season == 2025').copy()
    nm = playerid_reverse_lookup(s25.batter.tolist())
    nm['name'] = (nm.name_first + ' ' + nm.name_last).str.title()
    s25 = s25.merge(nm[['key_mlbam','name']], left_on='batter', right_on='key_mlbam')
    cols = ['name','pitches','decision_value']
    print('TOP 10, 2025');    display(s25.nlargest(10, 'decision_value')[cols].round(1))
    print('BOTTOM 10, 2025'); display(s25.nsmallest(10, 'decision_value')[cols].round(1))
except Exception as exc:
    print(f'name lookup unavailable ({exc})')

Gathering player lookup table. This may take a moment.


TOP 10, 2025


,name,pitches,decision_value
142,Leo Rivas,520,129.6
139,Ronald Acuña,1741,129.0
31,Max Muncy,1603,125.6
280,Davis Schneider,1012,125.2
116,Gleyber Torres,2652,125.1
97,Matt Thaiss,768,124.9
154,Kyle Tucker,2275,123.7
39,Aaron Judge,2631,121.1
180,Juan Soto,2968,120.2
301,Max Schuemann,838,120.1


BOTTOM 10, 2025


,name,pitches,decision_value
47,Javier Báez,1543,64.8
83,Edmundo Sosa,942,71.3
296,Jhonkensy Noel,563,72.0
326,Angel Martínez,1793,73.5
268,Lenyn Sosa,2056,74.3
286,Ronny Mauricio,692,75.1
120,Yohel Pozo,591,75.3
0,Martín Maldonado,569,76.2
256,Michael Harris,2173,78.0
81,Reese Mcguire,503,79.4


## Contamination — how much of the score is opportunity?

`signed_edge` pays more for an obvious take than for a close call (A1), so a
hitter thrown more junk can score higher for the same judgement. Raw Zone% |r|
is a screen for that, and a confounded one: better hitters are thrown fewer
strikes, so a metric that identifies good hitters picks up a *negative* Zone%
pull that can mask a positive bias.

So Zone% is also reported holding something fixed. Every control is a screen,
not proof:

- **correct-decision rate** is computed from the same model's Δ, and for
  `correct_decision` itself it is the metric, so that cell is meaningless;
- **chase and zone-swing rates** are model-free, but they vary with how hard the
  pitches a hitter sees are, so they do not hold judgement fixed either;
- **production** (ΔRE per PA) removes the "better hitters see fewer strikes"
  pull, which is what makes the raw test lenient.

The only test with ground truth is a known-policy simulation (Track B3).

In [20]:
key = ['season', 'batter']

def hitter_table(held):
    held = held.assign(inz=D.in_rulebook_zone(held, ball_edge=True), right=held.correct_decision > 0)
    g = held.groupby(key, observed=True)
    t = pd.DataFrame({'zone_pct': g.inz.mean(), 'cd_rate': g.right.mean(),
                      'chase': held[~held.inz].groupby(key, observed=True).swing.mean(),
                      'zone_swing': held[held.inz].groupby(key, observed=True).swing.mean()}).reset_index()
    return t.merge(E._production(held), on=key)

def partial(y, x, controls):
    Z = np.column_stack([np.ones(len(y))] + [np.asarray(c, float) for c in controls])
    res = lambda v: v - Z @ np.linalg.lstsq(Z, np.asarray(v, float), rcond=None)[0]
    return float(np.corrcoef(res(y), res(x))[0, 1])

CONTROLS = {'raw': [], '| correct-decision rate': ['cd_rate'],
            '| chase, zone-swing': ['chase', 'zone_swing'], '| production': ['re_per_pa']}

def contamination(scores, hit):
    """Zone% correlation per held-out season, averaged, under each control."""
    m = scores.merge(hit, on=key).dropna(subset=['decision_value', 'zone_pct', *hit.columns[2:]])
    return {f'Zone% r {label}': np.mean([partial(s.decision_value, s.zone_pct, [s[c] for c in ctrl])
                                         for _, s in m.groupby('season')])
            for label, ctrl in CONTROLS.items()}

hv3 = v3_run.held
hit_v3, hit_v1 = hitter_table(hv3), hitter_table(run_a1.held)
rows = []
for model, held, hit in [('v1 features', run_a1.held, hit_v1), ('v3', hv3, hit_v3)]:
    for score in ['chosen_value', 'signed_edge', 'regret', 'correct_decision']:
        rows.append({'model': model, 'score': score, **contamination(E.player_metric(held, score), hit)})
contam = pd.DataFrame(rows).set_index(['model', 'score'])
print(f"corr(Zone%, production) across hitter-seasons: {np.corrcoef(hit_v3.zone_pct, hit_v3.re_per_pa)[0, 1]:+.3f}")
contam.round(3)

corr(Zone%, production) across hitter-seasons: -0.225


Zone% r raw  Zone% r | correct-decision rate  Zone% r | chase, zone-swing  Zone% r | production
model       score                                                                                                            
v1 features chosen_value            0.082                           -0.253                       -0.442                 0.209
            signed_edge             0.301                            0.192                       -0.001                 0.398
            regret                  0.504                            0.562                        0.472                 0.569
            correct_decision        0.241                            0.052                        0.167                 0.326
v3          chosen_value           -0.180                           -0.417                       -0.496                -0.052
            signed_edge             0.184                           -0.006                       -0.236                 0.311
            regret                  0.547                            0.603                        0.527                 0.585
            correct_decision        0.213                            0.025                        0.066                 0.307

In [21]:
# Rescaling cannot help: z-score, an OPS+-style ratio and a percentile rank are
# monotone transforms of the same per-hitter mean.
sc = E.player_metric(hv3, 'signed_edge')
sc['ratio'] = sc.raw / sc.groupby('season').raw.transform('mean') * 100
sc['percentile'] = sc.groupby('season').raw.rank(pct=True) * 100
print(sc[['decision_value', 'ratio', 'percentile']].corr(method='spearman').round(4))
pd.DataFrame([{'scaling': col, **contamination(sc.assign(decision_value=sc[col]), hit_v3)}
              for col in ['decision_value', 'ratio', 'percentile']]).set_index('scaling').round(3)

                decision_value   ratio  percentile
decision_value          1.0000  0.9998      0.9997
ratio                   0.9998  1.0000      0.9996
percentile              0.9997  0.9996      1.0000


,Zone% r raw,Zone% r | correct-decision rate,"Zone% r | chase, zone-swing",Zone% r | production
scaling,,,,
decision_value,0.184,-0.006,-0.236,0.311
ratio,0.184,-0.006,-0.236,0.311
percentile,0.176,-0.008,-0.224,0.296


In [22]:
# Departure weighting: (swung - P(league swings | pitch)) x edge, so a pitch
# everyone handles the same way contributes ~0 to everyone. P(swing) is a
# league model fitted per fold on that fold's training seasons.
parts = []
for train_seasons, valid in E.PERSONALIZED_FOLDS:
    tr = pdf[pdf.season.isin(train_seasons)].dropna(subset=V3_FEATURES[3:])
    booster = xgb.train({**E.WHIFF_PARAMS, 'eval_metric': 'logloss'},
                        _dmatrix(tr, V3_FEATURES, tr.swing.astype(int)), 300)
    va = hv3[hv3.season == valid]
    parts.append(pd.Series(booster.predict(_dmatrix(va, V3_FEATURES)), index=va.index))
hv3['departure'] = (hv3.swing.astype(float) - pd.concat(parts)) * hv3.edge
hv3['region'] = region(hv3)
print(hv3.groupby('region', observed=True)[['signed_edge', 'departure']].mean().round(4), '\n')
pd.DataFrame({s: {**E.metric_checks(hv3, s)['summary'],
                  **contamination(E.player_metric(hv3, s), hit_v3)}
              for s in ['signed_edge', 'departure']}).round(3)

        signed_edge  departure
region                        
chase        0.0770     0.0011
shadow       0.0087     0.0006
zone         0.0488    -0.0001 



,signed_edge,departure
chase | zone_swing,-0.895,-0.902
zone_swing | chase,0.677,0.676
split-half r,0.838,0.860
YoY R2 (mean),0.624,0.658
Zone% |r|,0.184,0.323
next-season r (partial),0.154,0.098
Zone% r raw,0.184,0.323
Zone% r | correct-decision rate,-0.006,0.282
"Zone% r | chase, zone-swing",-0.236,0.092
Zone% r | production,0.311,0.419


In [23]:
# Opportunity standardization: a hitter's mean signed_edge within each stratum,
# reweighted to the league's mix of strata that season. Strata a hitter never
# saw take the league mean.
FAMILY = {**dict.fromkeys(['FF', 'SI', 'FC'], 'fastball'),
          **dict.fromkeys(['SL', 'ST', 'SV', 'CU', 'KC', 'CS'], 'breaking'),
          **dict.fromkeys(['CH', 'FS', 'FO', 'SC', 'KN', 'EP'], 'offspeed')}
hv3['family'] = hv3.pitch_type.astype(str).map(FAMILY).fillna('other')

def standardized(strata):
    cell = hv3.groupby(key + strata, observed=True).signed_edge.mean().rename('m').reset_index()
    league = hv3.groupby(['season'] + strata, observed=True).signed_edge.agg(lm='mean', n='size').reset_index()
    league['w'] = league.n / league.groupby('season').n.transform('sum')
    grid = hit_v3[key].merge(league, on='season').merge(cell, on=key + strata, how='left')
    grid['m'] = grid.m.fillna(grid.lm)
    s = (grid.w * grid.m).groupby([grid.season, grid.batter]).sum().rename('raw').reset_index()
    qualified = E.player_metric(hv3, 'signed_edge')[key]          # same >=500-pitch hitters
    s = s.merge(qualified, on=key)
    z = s.groupby('season').raw.transform(lambda v: (v - v.mean()) / v.std(ddof=0))
    return s.assign(decision_value=100 + 10 * z)

rows = [{'scoring': 'unadjusted', **contamination(E.player_metric(hv3, 'signed_edge'), hit_v3),
         **E.construct_validity(hv3, E.player_metric(hv3, 'signed_edge')).iloc[0][['chase | zone_swing', 'zone_swing | chase']]}]
for label, strata in [('region x count', ['region', 'count']),
                      ('region x count x pitch family', ['region', 'count', 'family'])]:
    s = standardized(strata)
    rows.append({'scoring': f'standardized: {label}', **contamination(s, hit_v3),
                 **E.construct_validity(hv3, s).iloc[0][['chase | zone_swing', 'zone_swing | chase']]})
pd.DataFrame(rows).set_index('scoring').T.round(3)

scoring,unadjusted,standardized: region x count,standardized: region x count x pitch family
Zone% r raw,0.184,0.236,0.217
Zone% r | correct-decision rate,-0.006,0.104,0.067
"Zone% r | chase, zone-swing",-0.236,-0.077,-0.095
Zone% r | production,0.311,0.359,0.343
chase | zone_swing,-0.895,-0.873,-0.865
zone_swing | chase,0.677,0.638,0.640


**The answer depends on which control you pick, so no control settles it.**
For v3's `signed_edge`, Zone% correlates +0.184 raw, −0.006 holding the
correct-decision rate fixed, −0.236 holding chase and zone-swing rates fixed,
and +0.311 holding production fixed.

**The raw test is lenient.** Better hitters are thrown fewer strikes
(corr(Zone%, production) = −0.225), so a score that tracks hitter quality picks
up a negative Zone% pull. Holding production fixed raises every score's Zone%
correlation, `signed_edge`'s from 0.184 to 0.311.

**`correct_decision` is not cleaner on the final model.** On v1's features it
is (raw 0.241 against 0.301; 0.326 against 0.398 holding production fixed). On
v3 the hot zone closes the gap: 0.213 against 0.184 raw, 0.307 against 0.311
holding production fixed. Its correct-decision-rate cell is circular and
meaningless.

**Rescaling cannot fix it.** Z-score, ratio and percentile correlate 0.9996 or
more with one another and give the same contamination.

**Departure weighting fails.** It does zero out the mean payout in every
region, and it is the most reliable score tested (split-half 0.860, YoY
0.658). But raw Zone% rises to 0.323, past the veto, and next-season validity
falls to 0.098.

**Opportunity standardization does not remove it.** Reweighting each hitter's
region × count means to the league mix — with or without pitch family — raises
raw Zone% (0.184 → 0.236 / 0.217) and the production-held figure (0.311 →
0.359 / 0.343), and costs construct validity. Only under the behaviour control
does it move toward zero. So the contamination survives *within* these strata:
the mix across them is not what drives it. That does not show pitch mix plays
no role — mix at a finer grain, such as location within a region or pitch
quality, is untested.

## Findings

All figures are held out: each of 2023–2025 scored by a fold model that never
saw it.

**What the patch buys**, against v1's features with the metric fixed:
next-season validity 0.082 → 0.154, Zone% contamination 0.300 → 0.184,
split-half reliability 0.818 → 0.838, YoY R² 0.594 → 0.624 — at a cost in
construct validity, −0.920 / +0.699 → −0.895 / +0.677.

**Personalization is the largest effect, and it works through the swing model
alone.** The continuous surface supplies most of the gain; the binary hull
supplies none. Adding the surface to the take model changes nothing.

**The metric is a trade, not a win.** By the criterion fixed before the numbers,
`correct_decision` is the best score at every feature set. `signed_edge` is
selected because it keeps the run-value magnitude and is the only candidate that
takes a substantial gain from the surface without collapsing construct validity. Its
pitch-mix contamination — Zone% |r| 0.30 without personalization, right at the
veto — is a documented limitation that the surface reduces to 0.18.

**The swing model learns location.** Held out, the batter frame recovers
90–94% of the location × count structure in swing value that a count-only
predictor misses, against 83–85% with raw location. The calibration slope is
above 1 at every step, suggesting the predictions understate the difference
between good and bad swing locations; it needs a confidence interval, and a
check grouped by predicted value, before it is a finding. Pitch-level RMSE,
0.8–0.9% better than count-only, is the wrong instrument for either.

**The take model is a called-strike probability.** Median R² 0.991 against a
fitted `P(CS)` on a held-out season, slopes matching the run-value gaps.

**Little in-sample optimism**: YoY R² 0.643 in-sample against 0.624 held out,
Zone% 0.223 against 0.184.

**Face validity holds.** The 2025 leaderboard puts Acuña, Muncy, Tucker, Judge
and Soto near the top and Báez at the bottom. The extremes include hitters just
over the 500-pitch line (Rivas, Thaiss), where the score is noisiest.

**For Track B.** Whiff probability predicts at AUC 0.77 held out, so the event
decomposition's premise holds. The take side is settled. Whatever the
swing model's calibration turns out to be, any direct-regression twin shares
it, so B0's calibration condition compares the twins with each other, not with
a slope of 1.

In [24]:
pd.DataFrame([v3['summary']]).set_index('variant')

,chase | zone_swing,zone_swing | chase,split-half r,YoY R2 (mean),Zone% |r|,next-season r (partial),swing bin err / count-only,swing signal recovered,swing calib slope,take bin err / count-only,take signal recovered,take calib slope
variant,,,,,,,,,,,,
v3,-0.895,0.677,0.838,0.624,0.184,0.154,0.0193 / 0.0365,0.892,1.291,0.0029 / 0.0363,0.994,1.014
